# Gaussian Process Regression for an Irregularly Sampled Stellar Light Curve
### Summer Student Program — Solutions Notebook

In this exercise you reconstruct an unknown stellar light curve from irregularly sampled, heteroscedastic observations using a Gaussian Process (GP). The notebook follows one continuous chain:

$$ k(t,t') \;\longrightarrow\; K_{ij}=k(t_i,t_j) \;\longrightarrow\; \text{prior functions} \;\longrightarrow\; p(\mathbf f_\star \mid \mathbf y) \;\longrightarrow\; \text{choose the next observation.} $$

| | |
|---|---|
| **Estimated time** | 60–75 minutes |
| **Packages** | `numpy`, `scipy`, `matplotlib` only — every GP operation is implemented by hand |
| **Scope** | zero mean function, fixed kernel hyperparameters (an optional section explores varying them) |

### Learning objectives
By the end you should be able to
1. interpret the amplitude $A$ and length scale $\ell$ of a squared-exponential kernel;
2. construct and inspect a GP covariance matrix;
3. draw functions from a GP prior;
4. implement the GP predictive mean and covariance with Cholesky linear algebra;
5. distinguish uncertainty in the latent signal from uncertainty in a future noisy observation;
6. use the predictive variance to choose a useful future observing time.

### Notation
$$ y_i = f(t_i) + \epsilon_i, \qquad \epsilon_i \sim \mathcal N(0,\sigma_i^2), \qquad f(t) \sim \mathcal{GP}\big(0,\, k(t,t')\big). $$

### How to use this notebook
- Cells marked `# TODO` contain blanks (`...`) for you to fill in. Plotting boilerplate is provided so you can concentrate on the GP.
- Several cells contain `assert` self-checks — if they pass, your implementation is correct.
- Questions marked **Q** should be answered in the markdown cell below them **before** moving on.
- The hidden true signal is revealed only in Part 6. Do not read the generator cell.


## 0 — Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cholesky, cho_solve, solve_triangular

plt.rcParams['figure.figsize'] = (10, 4.5)
plt.rcParams['font.size'] = 12

# np.trapezoid is numpy >= 2.0; fall back to np.trapz on older versions
trapezoid = getattr(np, 'trapezoid', None) or np.trapz

## Instructor-provided synthetic observations

The data set imitates a variable star observed at irregular times in **two observing seasons**, separated by a seasonal gap, with point-dependent measurement uncertainties and a smooth but non-trivial latent signal.

The generator is in the cell below. **Run it, but do not read `_hidden_stellar_signal`** — it is the answer you are trying to recover and is used only in the final validation section.

The data products are `t_obs` (days), `flux` (mean-subtracted relative flux) and `yerr` (1σ uncertainty per point).


In [ ]:
# ---- hidden generator: run but do not read -----------------------------------
rng_data = np.random.default_rng(2026)

t_obs = np.sort(np.concatenate([rng_data.uniform(0.3, 7.8, 32),
                                rng_data.uniform(12.2, 19.7, 28)]))

def _hidden_stellar_signal(t):
    """Synthetic latent stellar variability — used only for validation in Part 6."""
    t = np.asarray(t, dtype=float)
    return (0.65 * np.sin(2 * np.pi * t / 6.4 + 0.25)
            + 0.22 * np.sin(2 * np.pi * t / 2.5 - 0.7)
            + 0.12 * np.cos(2 * np.pi * t / 13.0)
            + 0.012 * (t - 10.0))

yerr = 0.07 + 0.08 * rng_data.random(t_obs.size)
yerr[rng_data.choice(t_obs.size, size=7, replace=False)] += 0.10        # a few poor-quality points

raw_flux = _hidden_stellar_signal(t_obs) + rng_data.normal(0.0, yerr)
flux_offset = np.average(raw_flux, weights=1.0 / yerr**2)             # centre so a zero GP mean is sensible
flux = raw_flux - flux_offset
# -------------------------------------------------------------------------------

print(f"N = {t_obs.size} observations, {t_obs.min():.2f}–{t_obs.max():.2f} d, median sigma = {np.median(yerr):.3f}")

---
# Part 1 — Inspect the astronomical time series

Before fitting anything, look at the sampling pattern and the uncertainties.

### Exercise 1.1
1. Plot the observations with error bars (points, not lines — connecting the dots already implies a model).
2. Compute the spacings `dt = np.diff(t_obs)` and find the start and end of the **longest gap**.
3. Shade the gap on the plot.


In [ ]:
dt = np.diff(t_obs)
i_gap = np.argmax(dt)
gap_start, gap_end = t_obs[i_gap], t_obs[i_gap + 1]
print(f"median dt = {np.median(dt):.2f} d,  longest gap: {gap_start:.2f}–{gap_end:.2f} d ({gap_end-gap_start:.1f} d)")

plt.errorbar(t_obs, flux, yerr=yerr, fmt='o', ms=4, capsize=2, color='k', label='observations')
plt.axvspan(gap_start, gap_end, color='C1', alpha=0.2, label='seasonal gap')
plt.xlabel('time [days]'); plt.ylabel('mean-subtracted relative flux'); plt.legend(); plt.grid(ls=':')

**Q1.1** Where is the largest gap, and how does it compare with the typical spacing?

*Your answer:*

> **Answer:** The largest gap runs from $t \approx 7.5$ to $12.3$ d (about 4.8 d), whereas the median spacing within a season is $\sim 0.2$ d. It is more than 20 times the typical spacing and, as we will see, several kernel length-scales long.


**Q1.2** Are all measurements equally precise? How should a model weight a point with $\sigma = 0.2$ compared with one with $\sigma = 0.08$?

*Your answer:*

> **Answer:** No: most points have $\sigma \approx 0.07$–$0.15$, and seven deliberately degraded points have $\sigma \approx 0.2$–$0.25$. In any Gaussian-likelihood fit the weight of a point scales as $1/\sigma^2$, so a $\sigma = 0.2$ point carries $(0.08/0.2)^2 \approx 1/6$ of the influence of a $\sigma = 0.08$ point; the model is allowed to pass further from it.


**Q1.3** Before computing anything: where do you expect the reconstruction uncertainty to be largest, and where smallest?

*Your answer:*

> **Answer:** Largest inside the seasonal gap and beyond the ends of the data (extrapolation), where no nearby observations constrain the function. Smallest where the sampling is dense and the error bars are small, typically the middle of each season. Near an isolated noisy point the uncertainty should be intermediate.


**Q1.4** Should a model of the latent stellar signal pass exactly through every point? Explain.

*Your answer:*

> **Answer:** No. Each $y_i = f(t_i) + \epsilon_i$ contains noise; a curve through every point would reproduce the noise (over-fitting) and would wiggle spuriously between points. The model should pass within $\sim 1\sigma_i$ of about two-thirds of the points. In the GP this is achieved by adding $\sigma_i^2$ to the diagonal of the training covariance.


---
# Part 2 — From a kernel to a GP prior

We use the squared-exponential kernel
$$ k(t,t') = A^2 \exp\!\left[-\frac{(t-t')^2}{2\ell^2}\right], $$
where $A$ is the typical vertical scale of the function ($A^2$ is its variance) and $\ell$ is the characteristic correlation time scale.

## Exercise 2.1 — Implement the squared-exponential kernel


In [ ]:
def squared_exponential_kernel(t1, t2, amplitude, length_scale):
    """Return the (len(t1), len(t2)) covariance matrix K[i, j] = k(t1[i], t2[j])."""
    if amplitude <= 0 or length_scale <= 0:
        raise ValueError("amplitude and length_scale must be positive")
    t1 = np.atleast_1d(np.asarray(t1, dtype=float))
    t2 = np.atleast_1d(np.asarray(t2, dtype=float))
    dt = t1[:, None] - t2[None, :]
    return amplitude**2 * np.exp(-0.5 * (dt / length_scale)**2)


# --- self-checks ---------------------------------------------------------------
K_test = squared_exponential_kernel(np.array([0.0, 1.0, 2.0]), np.array([0.0, 2.0]), amplitude=2.0, length_scale=1.0)
assert K_test.shape == (3, 2),                              "wrong shape"
assert np.isclose(K_test[0, 0], 4.0),                       "k(t,t) must equal A^2"
assert np.isclose(K_test[1, 0], 4.0 * np.exp(-0.5)),        "k at one length-scale must be A^2 e^{-1/2}"
assert np.isclose(K_test[0, 1], K_test[2, 0]),              "k must depend only on |t - t'|"
print("all checks passed")

## Exercise 2.2 — Plot and interpret the covariance function

Because the kernel depends only on the separation $\tau = t - t'$, plot $k(\tau)$ for $\tau \in [-5, 5]$ d using the fixed values $A = 0.7$, $\ell = 1.1$ d that we will use throughout. Mark $\pm\ell$ with vertical lines and the points $k(0)$ and $k(\ell)$.


In [ ]:
amplitude, length_scale = 0.7, 1.1          # fixed hyperparameters for the whole notebook
tau = np.linspace(-5.0, 5.0, 500)

k_tau = squared_exponential_kernel(tau, [0.0], amplitude, length_scale)[:, 0]
k0, k_ell = amplitude**2, amplitude**2 * np.exp(-0.5)

plt.plot(tau, k_tau)
plt.axvline(-length_scale, ls='--', color='0.5'); plt.axvline(length_scale, ls='--', color='0.5', label=r'$\pm\ell$')
plt.plot(0, k0, 'ko'); plt.annotate(rf'$k(0)=A^2={k0:.3f}$', (0, k0), xytext=(0.4, k0))
plt.plot(length_scale, k_ell, 'rs'); plt.annotate(rf'$k(\ell)={k_ell:.3f}$', (length_scale, k_ell), xytext=(length_scale + 0.4, k_ell))
plt.xlabel(r'separation $\tau = t - t^{\prime}$ [days]'); plt.ylabel(r'$k(\tau)$'); plt.legend(); plt.grid(ls=':')
print(f"k(ell) / k(0) = {k_ell / k0:.3f}")

**Q2.2** (a) What is $k(t,t)$? (b) What fraction of $k(t,t)$ remains at $|t-t'| = \ell$, and at $3\ell$? (c) Which parameter controls vertical variability and which controls how quickly the function can change?

*Your answer:*

> **Answer:** (a) $k(t,t) = A^2 = 0.49$, the prior variance of $f$ at any time. (b) $k(\ell)/k(0) = e^{-1/2} \approx 0.61$; at $3\ell$ it is $e^{-4.5} \approx 0.011$ — essentially uncorrelated. (c) $A$ sets the vertical scale (functions live within roughly $\pm 2A$); $\ell$ sets the horizontal scale: two values closer than $\sim\ell$ are strongly correlated, so the function cannot change much over that interval.


## Exercise 2.3 — Construct and inspect a covariance matrix

Evaluate $K_{ij} = k(t_i, t_j)$ on a regular grid of 150 times on $[0, 20]$ d and display it as an image. Then check numerically that $K = K^{\mathsf T}$ and that its eigenvalues are non-negative up to floating-point precision.


In [ ]:
t_cov = np.linspace(0.0, 20.0, 150)
K_cov = squared_exponential_kernel(t_cov, t_cov, amplitude, length_scale)

plt.figure(figsize=(6, 5))
im = plt.imshow(K_cov, origin='lower', extent=[0, 20, 0, 20], cmap='viridis')
plt.colorbar(im, label=r'$K_{ij}$'); plt.xlabel(r'$t_j$ [days]'); plt.ylabel(r'$t_i$ [days]'); plt.title(r'$K_{ij}=k(t_i,t_j)$')

symmetry_error = np.max(np.abs(K_cov - K_cov.T))
eigenvalues = np.linalg.eigvalsh(K_cov)
print(f"max |K - K^T|      = {symmetry_error:.2e}")
print(f"smallest eigenvalue = {eigenvalues.min():.2e}   largest = {eigenvalues.max():.2e}")
print(f"eigenvalues > 1e-10 : {np.sum(eigenvalues > 1e-10)} of {eigenvalues.size}")

**Q2.3** (a) Why is the diagonal the brightest region? (b) Why is there a band rather than only a thin diagonal line? (c) What would happen to the band if $\ell$ were larger? (d) Why is a tiny negative eigenvalue of order $10^{-15}$ not a physical violation, and what does it mean in practice?

*Your answer:*

> **Answer:** (a) On the diagonal $t_i = t_j$, so $K_{ii} = k(0) = A^2$, the maximum; it is the variance of $f(t_i)$, and a value is perfectly correlated with itself. (b) Neighbouring times are still strongly correlated — $K_{ij}$ only falls to $0.61 A^2$ at $|t_i - t_j| = \ell$ — so the bright region extends $\sim\ell$ either side of the diagonal. (c) A larger $\ell$ widens the band; for $\ell \gtrsim 20$ d almost the whole matrix would be bright. (d) Mathematically $K$ is positive semi-definite, but on a dense grid many rows are nearly identical, so the true smallest eigenvalues are $\sim 0$ and floating-point rounding can push them slightly negative. Only about 20 of the 150 eigenvalues are above $10^{-10}$: the matrix is numerically rank-deficient. In practice we add a tiny *jitter* $\epsilon I$ to the diagonal before a Cholesky factorisation.


## Exercise 2.4 — Draw sample functions from the GP prior

A GP prior says that the vector of function values on a grid is a draw from $\mathcal N(\mathbf 0, K)$. To generate one, factorise $K + \epsilon I = L L^{\mathsf T}$ (Cholesky) and set
$$ \mathbf f = L \mathbf z, \qquad \mathbf z \sim \mathcal N(\mathbf 0, I). $$
(Check: $\mathrm{Cov}[L\mathbf z] = L L^{\mathsf T} = K$.)


In [ ]:
def draw_gp_prior(t, amplitude, length_scale, n_samples=4, rng=None, jitter=1e-10):
    """Return an array of shape (len(t), n_samples) of functions drawn from the zero-mean GP prior."""
    t = np.atleast_1d(np.asarray(t, dtype=float))
    if rng is None:
        rng = np.random.default_rng()
    K = squared_exponential_kernel(t, t, amplitude, length_scale)
    L = cholesky(K + jitter * np.eye(t.size), lower=True)
    z = rng.normal(size=(t.size, n_samples))
    return L @ z


t_prior = np.linspace(0.0, 20.0, 300)
prior_samples = draw_gp_prior(t_prior, amplitude, length_scale, n_samples=4, rng=np.random.default_rng(7))
assert prior_samples.shape == (300, 4)

plt.plot(t_prior, prior_samples, lw=1.3)
plt.fill_between(t_prior, -2 * amplitude, 2 * amplitude, color='0.85', alpha=0.5, label=r'prior $\pm 2A$')
plt.xlabel('time [days]'); plt.ylabel('latent flux'); plt.title('Four independent samples from the same GP prior'); plt.legend(); plt.grid(ls=':')

**Q2.4** (a) Why do the curves have different detailed shapes? (b) What statistical properties do they share? (c) Why do they look like coherent curves rather than independent random points?

*Your answer:*

> **Answer:** (a) Each is a different random realisation — a different $\mathbf z$ — from the same distribution; the kernel defines a distribution over functions, not one function. (b) Zero mean, the same pointwise variance $A^2$ (so they stay mostly within $\pm 2A$), the same correlation-vs-lag structure and hence the same typical wiggle time scale $\sim\ell$, and the same smoothness. (c) Because neighbouring grid points are strongly correlated through the off-diagonal elements of $K$: the Cholesky factor $L$ mixes the independent $z_i$ so that $f(t_{i+1})$ is almost equal to $f(t_i)$. With a diagonal $K$ (white noise) the samples *would* be independent points.


## Exercise 2.5 — Change the length scale

Draw three prior samples each for $\ell = 0.35$ d and $\ell = 2.5$ d, keeping $A = 0.7$ and using the **same random numbers** (same `rng` seed) in both panels so that only the kernel changes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, ell_trial in zip(axes, [0.35, 2.5]):
    samples = draw_gp_prior(t_prior, amplitude, ell_trial, n_samples=3, rng=np.random.default_rng(8))
    ax.plot(t_prior, samples, lw=1.3)
    ax.fill_between(t_prior, -2 * amplitude, 2 * amplitude, color='0.85', alpha=0.5)
    ax.set_title(rf'$A={amplitude},\ \ell={ell_trial}$ d'); ax.set_xlabel('time [days]'); ax.grid(ls=':')
axes[0].set_ylabel('latent flux')

**Q2.5** Explain why reducing $\ell$ makes the functions vary more rapidly even though $A$ is unchanged. Which value of $\ell$ looks more like the light curve from Part 1?

*Your answer:*

> **Answer:** $\ell$ sets the lag over which correlations decay. With $\ell = 0.35$ d, values more than $\sim 1$ d apart are independent, so the function can reverse direction many times in 20 days; with $\ell = 2.5$ d it must stay coherent over several days and can complete only a few slow bumps. The *size* of the excursions is still set by $A$ — both panels fill the same $\pm 2A$ band. The light curve in Part 1 has bumps lasting a few days, so a value near 1 d (our choice of 1.1 d) is more plausible than either extreme.


---
# Part 3 — Condition the GP on the observations

For a zero mean function, the predictive distribution of the latent function at test times $\mathbf t_\star$ is
$$ p(\mathbf f_\star \mid \mathbf y) = \mathcal N(\boldsymbol\mu_\star, \mathbf C_\star), \qquad
\boldsymbol\mu_\star = \mathbf K_\star^{\mathsf T} \mathbf K_y^{-1} \mathbf y, \qquad
\mathbf C_\star = \mathbf K_{\star\star} - \mathbf K_\star^{\mathsf T} \mathbf K_y^{-1} \mathbf K_\star, $$
with $[\mathbf K_y]_{ij} = k(t_i,t_j) + \sigma_i^2 \delta_{ij}$, $[\mathbf K_\star]_{ij} = k(t_i, t_{\star j})$ and $[\mathbf K_{\star\star}]_{ij} = k(t_{\star i}, t_{\star j})$.

We never form $\mathbf K_y^{-1}$ explicitly. With the Cholesky factor $\mathbf K_y = L L^{\mathsf T}$:
$$ \boldsymbol\alpha = \mathbf K_y^{-1}\mathbf y \;(\texttt{cho\_solve}), \qquad
\boldsymbol\mu_\star = \mathbf K_\star^{\mathsf T}\boldsymbol\alpha, \qquad
\mathbf v = L^{-1}\mathbf K_\star \;(\texttt{solve\_triangular}), \qquad
\mathbf C_\star = \mathbf K_{\star\star} - \mathbf v^{\mathsf T}\mathbf v . $$

## Exercise 3.1 — Implement GP prediction


In [ ]:
def gp_predict(t_train, y_train, yerr_train, t_test, amplitude, length_scale, jitter=1e-10):
    """Return the posterior mean (n_test,) and covariance (n_test, n_test) of the latent function."""
    t_train = np.atleast_1d(np.asarray(t_train, dtype=float))
    y_train = np.atleast_1d(np.asarray(y_train, dtype=float))
    yerr_train = np.atleast_1d(np.asarray(yerr_train, dtype=float))
    t_test = np.atleast_1d(np.asarray(t_test, dtype=float))
    if not (t_train.size == y_train.size == yerr_train.size):
        raise ValueError("t_train, y_train and yerr_train must have equal length")

    K_y = squared_exponential_kernel(t_train, t_train, amplitude, length_scale) + np.diag(yerr_train**2 + jitter)
    K_star = squared_exponential_kernel(t_train, t_test, amplitude, length_scale)
    K_starstar = squared_exponential_kernel(t_test, t_test, amplitude, length_scale)
    L = cholesky(K_y, lower=True)
    alpha = cho_solve((L, True), y_train)
    posterior_mean = K_star.T @ alpha
    v = solve_triangular(L, K_star, lower=True)
    posterior_covariance = K_starstar - v.T @ v
    posterior_covariance = 0.5 * (posterior_covariance + posterior_covariance.T)   # remove round-off asymmetry
    return posterior_mean, posterior_covariance


# --- self-checks ---------------------------------------------------------------
m, Cv = gp_predict(t_obs[:8], flux[:8], yerr[:8], np.linspace(0.0, 4.0, 25), amplitude, length_scale)
assert m.shape == (25,) and Cv.shape == (25, 25),      "wrong output shapes"
assert np.all(np.diag(Cv) <= amplitude**2 + 1e-8),     "posterior variance can never exceed the prior variance A^2"
# a single exact (sigma -> 0) observation must be reproduced at that time
m1, C1 = gp_predict([3.0], [0.4], [1e-6], [3.0, 30.0], amplitude, length_scale)
assert np.isclose(m1[0], 0.4, atol=1e-4) and np.isclose(C1[0, 0], 0.0, atol=1e-4), "exact data must be interpolated exactly"
assert np.isclose(m1[1], 0.0) and np.isclose(C1[1, 1], amplitude**2),               "far from data, revert to the prior"
print("all checks passed")

## Exercise 3.2 — Reconstruct the light curve

Evaluate the posterior on a fine grid `t_prediction` over $[0, 20]$ d with the fixed values $A = 0.7$, $\ell = 1.1$ d (supplied rather than optimised — hyperparameter training is outside the scope of the main exercise; see the optional Part 7). Extract the posterior standard deviation from the diagonal of the covariance and plot the observations, the posterior mean and the latent $\pm1\sigma$ and $\pm2\sigma$ intervals.


In [ ]:
t_prediction = np.linspace(0.0, 20.0, 500)

posterior_mean, posterior_covariance = gp_predict(t_obs, flux, yerr, t_prediction, amplitude, length_scale)
latent_std = np.sqrt(np.clip(np.diag(posterior_covariance), 0.0, None))
print(f"posterior std ranges from {latent_std.min():.3f} to {latent_std.max():.3f}  (prior: {amplitude:.3f})")


def plot_posterior(mean, std, label='posterior mean', ax=None):
    """Helper used repeatedly below."""
    ax = ax or plt.gca()
    ax.fill_between(t_prediction, mean - 2 * std, mean + 2 * std, color='C0', alpha=0.15, label=r'latent $\pm2\sigma$')
    ax.fill_between(t_prediction, mean - std, mean + std, color='C0', alpha=0.3, label=r'latent $\pm1\sigma$')
    ax.plot(t_prediction, mean, 'C0-', lw=1.5, label=label)
    ax.errorbar(t_obs, flux, yerr=yerr, fmt='ko', ms=3, capsize=2, label='observations')
    ax.axvspan(gap_start, gap_end, color='C1', alpha=0.12)
    ax.set_xlabel('time [days]'); ax.set_ylabel('mean-subtracted relative flux'); ax.grid(ls=':')
    return ax

plot_posterior(posterior_mean, latent_std).legend(loc='upper right', ncol=2)
plt.title('GP reconstruction of the latent stellar light curve')

**Q3.2** (a) Where is the posterior uncertainty largest — does it match your prediction in Q1.3? (b) Why does the posterior mean not pass through every observation? (c) What happens to the prediction far from informative observations? (d) Why can a GP be confident between two closely spaced precise measurements but less confident near a single noisy one?

*Your answer:*

> **Answer:** (a) Largest in the middle of the seasonal gap (std $\to$ the prior value 0.7 by $t \approx 10$ d) and beyond $t \approx 19.3$ d; smallest ($\approx 0.04$) in the densely sampled parts of each season. This matches Q1.3. (b) The $\sigma_i^2$ terms on the diagonal of $\mathbf K_y$ let the mean deviate from each point by an amount comparable to its error bar; the noisy ($\sigma \approx 0.2$) points are visibly not followed. (c) The mean relaxes to the prior mean (zero) and the variance to the prior $A^2$ within a few length-scales: the GP 'forgets'. (d) Two precise points $\lesssim \ell$ apart pin down both the value and the slope of $f$ in between, and their information adds; a single noisy point gives only a weak constraint on the value and none on the slope, so the posterior there remains broad.


## Exercise 3.3 — Draw functions from the posterior

Exactly as in Part 2, but now with the posterior mean and covariance: $\mathbf f = \boldsymbol\mu_\star + L_\star \mathbf z$ with $L_\star L_\star^{\mathsf T} = \mathbf C_\star + \epsilon I$. Draw four samples and overplot the observations.


In [ ]:
rng_posterior = np.random.default_rng(11)
L_post = cholesky(posterior_covariance + 1e-9 * np.eye(t_prediction.size), lower=True)
posterior_samples = posterior_mean[:, None] + L_post @ rng_posterior.normal(size=(t_prediction.size, 4))

plt.plot(t_prediction, posterior_samples, lw=1.2)
plt.errorbar(t_obs, flux, yerr=yerr, fmt='ko', ms=3, capsize=2, label='observations')
plt.axvspan(gap_start, gap_end, color='C1', alpha=0.12)
plt.xlabel('time [days]'); plt.ylabel('mean-subtracted relative flux'); plt.title('Functions sampled from the GP posterior'); plt.legend(); plt.grid(ls=':')

**Q3.3** Why do the posterior curves agree closely in some regions but differ substantially in the seasonal gap? What extra information do the samples give that the $\pm 2\sigma$ band does not?

*Your answer:*

> **Answer:** Where data are dense, $\mathbf C_\star$ is small and all draws collapse onto the posterior mean; in the gap $\mathbf C_\star$ returns toward the prior and each draw follows its own prior-like excursion. The band only shows the *marginal* uncertainty at each time; the samples also show the *correlations* between times — each is a complete, internally consistent light curve with the right smoothness, not a collection of independent points. Any quantity that depends on the whole curve (e.g. the time of the next minimum) must be estimated from samples, not from the band.


---
# Part 4 — Latent-function uncertainty versus future-observation uncertainty

The covariance computed above describes uncertainty in the **latent function** $f_\star$. A future *measured* value is $y_\star = f_\star + \epsilon_\star$, so
$$ \mathrm{Var}(y_\star \mid \mathbf y) = \mathrm{Var}(f_\star \mid \mathbf y) + \sigma_\star^2 . $$

Assume a future measurement has the median uncertainty of the existing data. Compute the standard deviation for a future observation and plot its $\pm 2\sigma$ interval together with the latent one.


In [ ]:
sigma_future = float(np.median(yerr))
future_observation_std = np.sqrt(latent_std**2 + sigma_future**2)

ax = plot_posterior(posterior_mean, latent_std)
ax.fill_between(t_prediction, posterior_mean - 2 * future_observation_std, posterior_mean + 2 * future_observation_std,
                color='C3', alpha=0.12, label=r'future observation $\pm2\sigma$')
ax.legend(loc='upper right', ncol=2); plt.title('Two different predictive uncertainties')
print(f"assumed sigma of a future measurement: {sigma_future:.3f}")

**Q4** (a) Why is the future-observation interval wider, and where is the difference most visible? (b) Which interval should be reported for the underlying stellar variability? (c) Which should be used to forecast a future detector measurement, e.g. to decide whether a new point is anomalous?

*Your answer:*

> **Answer:** (a) It adds a fresh realisation of measurement noise, $\sigma_\star^2$, in quadrature. The difference is most visible where the latent uncertainty is small (inside the seasons, where the red band is much wider than the blue one) and negligible in the gap, where the latent uncertainty already dominates. (b) The latent interval — that is our knowledge of the star. (c) The future-observation interval; a new measurement falling outside the *latent* band is expected roughly a third of the time and is not anomalous.


---
# Part 5 — Choose the next observation

For fixed kernel hyperparameters the posterior covariance $\mathbf C_\star$ depends on the observation **times and uncertainties** but not on the measured **values** (look at the formula: $\mathbf y$ appears in $\boldsymbol\mu_\star$ only). We can therefore use the predictive variance to schedule a new observation before taking it. We adopt the simplest acquisition rule,
$$ t_{\rm next} = \operatorname*{arg\,max}_t \mathrm{Var}[f(t) \mid \mathbf y], $$
excluding candidate times within 0.25 d of an existing observation.

## Exercise 5.1 — Select the time


In [ ]:
candidate_times = np.linspace(0.5, 19.5, 400)
nearest = np.min(np.abs(candidate_times[:, None] - t_obs[None, :]), axis=1)
candidate_times = candidate_times[nearest > 0.25]

_, C_cand = gp_predict(t_obs, flux, yerr, candidate_times, amplitude, length_scale)
candidate_variance = np.clip(np.diag(C_cand), 0.0, None)
t_next = float(candidate_times[np.argmax(candidate_variance)])
print(f"selected next observing time: {t_next:.3f} d")

plt.plot(candidate_times, candidate_variance, '.', ms=3, label=r'$\mathrm{Var}[f(t)\,|\,\mathbf{y}]$')
plt.axvline(t_next, ls='--', color='C3', label=f'selected: {t_next:.2f} d')
plt.axhline(amplitude**2, ls=':', color='0.5', label=r'prior variance $A^2$')
plt.xlabel('candidate observing time [days]'); plt.ylabel('posterior variance'); plt.legend(); plt.grid(ls=':')

**Q5.1** (a) Where was the next observation selected and why? (b) The variance curve has a flat top: what does that tell you about the choice? (c) Would changing only the measured flux values change the selected time? Verify your answer with a one-line experiment (e.g. call `gp_predict` with `2 * flux` or with `flux[::-1]`).

*Your answer:*

> **Answer:** (a) Near $t \approx 10$ d, the centre of the seasonal gap, where the nearest data are $\sim 2.5$ d $\approx 2.3\ell$ away on either side and the variance has returned to the prior value $A^2$. (b) The variance saturates at $A^2$ over roughly $9$–$11$ d: every time in that range is equally uninformed, so the argmax is essentially arbitrary within it — a real scheduler would break the tie with other criteria. (c) No: $\mathbf C_\star$ does not contain $\mathbf y$. Calling `gp_predict(t_obs, 2*flux, yerr, ...)` returns the identical covariance (and a doubled mean). This would *not* hold if the hyperparameters were re-fitted to the new values.


## Exercise 5.2 — Add the new measurement and update the GP

The cell below simulates one noisy measurement at `t_next` (the hidden signal is used here only to generate a realistic datum). Append it to the data, recompute the posterior, and compare the posterior standard deviation and the **integrated posterior variance** $\int \mathrm{Var}[f(t)\mid\mathbf y]\,dt$ before and after.


In [ ]:
sigma_next = float(np.median(yerr))
flux_next = float(_hidden_stellar_signal(t_next) + np.random.default_rng(17).normal(0.0, sigma_next) - flux_offset)
print(f"new simulated observation: t = {t_next:.3f}, y = {flux_next:.3f}, sigma = {sigma_next:.3f}")

In [ ]:
t_aug, flux_aug, yerr_aug = np.append(t_obs, t_next), np.append(flux, flux_next), np.append(yerr, sigma_next)
order = np.argsort(t_aug)
t_aug, flux_aug, yerr_aug = t_aug[order], flux_aug[order], yerr_aug[order]

mean_aug, cov_aug = gp_predict(t_aug, flux_aug, yerr_aug, t_prediction, amplitude, length_scale)
std_aug = np.sqrt(np.clip(np.diag(cov_aug), 0.0, None))

int_var_before = trapezoid(latent_std**2, t_prediction)
int_var_after  = trapezoid(std_aug**2, t_prediction)
print(f"integrated posterior variance: before {int_var_before:.3f}, after {int_var_after:.3f}, fraction remaining {int_var_after/int_var_before:.3f}")

plt.plot(t_prediction, latent_std, label='before new observation')
plt.plot(t_prediction, std_aug, label='after new observation')
plt.axvline(t_next, ls='--', color='C3', label='new observation')
plt.xlabel('time [days]'); plt.ylabel('posterior standard deviation'); plt.title('Effect of one targeted observation'); plt.legend(); plt.grid(ls=':')

**Q5.2** (a) Where did the uncertainty decrease most strongly, and over what range of times? How is that range related to $\ell$? (b) By how much did the integrated variance drop? (c) Why might a real observing programme use a criterion more sophisticated than maximum variance alone?

*Your answer:*

> **Answer:** (a) At $t_{\rm next} \approx 10$ d the standard deviation drops from 0.70 to about the measurement error, 0.12. The reduction extends $\sim \pm 2\ell \approx \pm 2$ d on either side, because the new point informs $f$ only over the kernel's correlation length. (b) The integrated variance falls to $\approx 40\%$ of its previous value — a single well-placed point removes 60% of the total uncertainty, because the gap dominated the integral. (c) Maximum variance ignores everything else: the *cost* of observing (a point in the gap may be unobservable — that is often why the gap exists), visibility and weather, exposure-time trade-offs, the fact that the variance plateau makes the exact time arbitrary, and the *scientific* goal (one might care about the time of a minimum rather than the whole curve, which calls for a different acquisition function). It also assumes the kernel is correct; if the star is really periodic, a periodic kernel would already predict the gap and the best new point would be elsewhere.


---
# Part 6 — Reveal the latent signal and assess the reconstruction

Only now do we compare the GP reconstruction with the synthetic truth. Plot the truth (minus `flux_offset`), the posterior mean before and after the new observation, the updated $\pm 2\sigma$ band and all observations; then compute the RMSE of the posterior mean against the truth over the full grid and inside the gap, before and after.


In [ ]:
true_signal = _hidden_stellar_signal(t_prediction) - flux_offset

ax = plot_posterior(mean_aug, std_aug, label='posterior mean after')
ax.plot(t_prediction, posterior_mean, 'C2-', lw=1.3, label='posterior mean before')
ax.plot(t_prediction, true_signal, 'C3--', lw=1.5, label='synthetic truth')
ax.errorbar([t_next], [flux_next], yerr=[sigma_next], fmt='o', color='C3', ms=7, capsize=3, label='new observation')
ax.legend(loc='upper right', ncol=2); plt.title('GP reconstruction compared with the synthetic truth')

gap_mask = (t_prediction > gap_start) & (t_prediction < gap_end)
rmse = lambda a, b: np.sqrt(np.mean((a - b)**2))
print(f"overall RMSE: before {rmse(posterior_mean, true_signal):.3f}   after {rmse(mean_aug, true_signal):.3f}")
print(f"gap RMSE:     before {rmse(posterior_mean[gap_mask], true_signal[gap_mask]):.3f}   after {rmse(mean_aug[gap_mask], true_signal[gap_mask]):.3f}")

# calibration: fraction of the truth inside the ±2σ band
print(f"truth inside latent 2σ band: before {np.mean(np.abs(true_signal - posterior_mean) < 2*latent_std):.2f}   after {np.mean(np.abs(true_signal - mean_aug) < 2*std_aug):.2f}   (expect ~0.95)")

**Q6.1** Did the new observation improve the RMSE in this realisation — overall and in the gap? Is the $\pm 2\sigma$ band well calibrated?

*Your answer:*

> **Answer:** Yes: overall RMSE drops from $\approx 0.16$ to $\approx 0.12$ and the gap RMSE from $\approx 0.30$ to $\approx 0.21$. The truth lies inside the $\pm 2\sigma$ band for $\gtrsim 95\%$ of the grid, so the band is about right — the fixed $A$, $\ell$ happen to be reasonable for this star (the truth does leave the band briefly at the very start, where the model extrapolates).


**Q6.2** Why is a reduction in posterior *variance* a more reliable observing-design objective than a guaranteed reduction in RMSE?

*Your answer:*

> **Answer:** The variance reduction is known *before* the observation is taken (it depends only on $t_{\rm next}$ and $\sigma_{\rm next}$) and is guaranteed to be non-negative. The RMSE against the truth depends on the particular noise realisation of the new point: a noisy measurement that happens to fall far from the truth can pull the mean the wrong way and *increase* the RMSE locally, even though on average over many realisations the expected squared error equals the posterior variance and therefore decreases. We can plan on expectations, not on one draw.


**Q6.3** State in one sentence each: what the kernel contributes to this analysis, and what conditioning on the observations changes.

*Your answer:*

> **Answer:** The kernel encodes our prior beliefs about the statistical character of the light curve — its typical amplitude, smoothness and correlation time — and thereby defines a distribution over functions before any data are seen. Conditioning on the observations narrows that distribution to the functions consistent with the data: it moves the mean from zero to the reconstruction and shrinks the covariance wherever data (weighted by their errors) are informative, leaving it at the prior elsewhere.


---
# Part 7 (optional) — What if the hyperparameters are wrong?

The values $A = 0.7$, $\ell = 1.1$ d were handed to you. Everything above depends on them. Refit the posterior with $\ell = 0.35$ d and $\ell = 2.5$ d (keeping $A = 0.7$) and compare the posterior mean and band with the truth. Then compute the **log marginal likelihood** for a grid of $\ell$ values,
$$ \log p(\mathbf y \mid \ell) = -\tfrac12 \mathbf y^{\mathsf T}\mathbf K_y^{-1}\mathbf y - \sum_i \log L_{ii} - \tfrac{N}{2}\log 2\pi , $$
(with $L$ the Cholesky factor of $\mathbf K_y$) and see which $\ell$ the data prefer.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, ell_trial in zip(axes, [0.35, 1.1, 2.5]):
    m_, C_ = gp_predict(t_obs, flux, yerr, t_prediction, amplitude, ell_trial)
    plot_posterior(m_, np.sqrt(np.clip(np.diag(C_), 0, None)), ax=ax)
    ax.plot(t_prediction, true_signal, 'C3--', lw=1.2, label='truth')
    ax.set_title(rf'$\ell = {ell_trial}$ d'); ax.set_ylim(-1.8, 1.8)
axes[0].legend(loc='lower left', fontsize=9)


def log_marginal_likelihood(t_train, y_train, yerr_train, amplitude, length_scale, jitter=1e-10):
    K_y = squared_exponential_kernel(t_train, t_train, amplitude, length_scale) + np.diag(yerr_train**2 + jitter)
    L = cholesky(K_y, lower=True)
    alpha = cho_solve((L, True), y_train)
    return -0.5 * y_train @ alpha - np.sum(np.log(np.diag(L))) - 0.5 * y_train.size * np.log(2 * np.pi)

ells = np.logspace(np.log10(0.2), np.log10(5.0), 60)
lml = np.array([log_marginal_likelihood(t_obs, flux, yerr, amplitude, ell) for ell in ells])
plt.figure(); plt.semilogx(ells, lml); plt.axvline(1.1, ls='--', color='0.5', label='fixed value 1.1 d')
plt.axvline(ells[np.argmax(lml)], ls=':', color='C3', label=f'maximum: {ells[np.argmax(lml)]:.2f} d')
plt.xlabel(r'$\ell$ [days]'); plt.ylabel('log marginal likelihood'); plt.legend(); plt.grid(ls=':', which='both')

**Q7** Describe the failure modes at $\ell = 0.35$ d and $\ell = 2.5$ d. Does the marginal likelihood prefer a value close to the one you were given? Why does a *too small* $\ell$ produce small residuals but a poor reconstruction?

*Your answer:*

> **Answer:** With $\ell = 0.35$ d the mean chases individual noisy points and the band between points (and across the gap) is almost the full prior width: the model over-fits the noise and under-uses neighbouring data. With $\ell = 2.5$ d the mean is too smooth to follow the 2.5-day harmonic, visibly missing the sharper features, and the band is over-confident (the truth leaves it). The marginal likelihood peaks at $\ell \approx 0.8$ d — close to, but slightly below, the supplied 1.1 d (the 2.5-day harmonic in the truth pulls the preferred scale down), and the curve is flat-topped between $\sim 0.5$ and $1.2$ d, so both values are acceptable while 0.35 d and 2.5 d are strongly disfavoured. It balances data fit (penalising the over-smooth model) against model complexity via the $\sum\log L_{ii}$ term (penalising the over-flexible one). Small $\ell$ gives small residuals precisely because it can fit anything — but fitting the noise carries no information about $f$ between the points.


---
# Take-home messages
- The kernel specifies statistical relationships between function values; it does not specify one unique curve.
- A covariance matrix is obtained by evaluating the kernel for every pair of input times; it is symmetric and positive semi-definite, and a small jitter keeps it numerically so.
- Prior functions share common statistical properties but differ as random realisations.
- Conditioning on observations gives a posterior mean and a posterior covariance, both computed with one Cholesky factorisation and triangular solves — never an explicit inverse.
- Measurement noise belongs in the training covariance; it is added again only when predicting a future *noisy* observation.
- Predictive variance depends on where and how well we observed, not on what we measured — which is exactly what makes it usable for planning observations.
- The hyperparameters are assumptions; check them (marginal likelihood, held-out data) before trusting the band.

### Source basis
This exercise is based on the Gaussian Process lecture materials supplied with the course, especially their treatment of covariance functions, prior samples, predictive equations, white noise, and active sampling.
